# My RL Environment Exploration: Building a Coding Task Loop

I am exploring how to build an environment that can generate model rollouts, check their outputs, and return scores. I will start with a small Python coding task set, implement the loop myself, then compare it with a Verifiers-based version.

## Project questions

1. What are the core parts of an RL environment?
2. How can I run generated code and score it against tests?
3. How do parsers affect the reliability of a rubric?
4. What changes when I let the model use feedback across multiple turns?
5. Can one environment support evaluation, training signals, and synthetic data collection?

My working model is **dataset + rollout + rubric**: the dataset supplies tasks, the rollout defines how the model interacts with them, and the rubric scores the result. I will test how this structure behaves in a small coding project.

---
## 0. Set up the experiment

I need an OpenRouter API key for model calls, the `openai` SDK for its compatible endpoint, and `datasets` for the later Verifiers example. I will install Verifiers in the section where I use it.

In [ ]:
# Install the dependencies I need for this experiment.
!pip install -q openai datasets


In [ ]:
import os
import json
import subprocess
import tempfile
import textwrap
from google.colab import userdata
from typing import Any, Callable



from openai import OpenAI

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=userdata.get('OPENROUTER_API_KEY'),
)

# Check that the API connection responds before I build the environment.
resp = client.chat.completions.create(
    model="google/gemini-2.0-flash-001",
    messages=[{"role": "user", "content": "Reply with exactly the word 'pong'."}],
    max_tokens=10,
)
print("Model said:", resp.choices[0].message.content.strip())


Model said: pong


---
## 1. My mental model of an environment

I am treating an LLM environment as three connected components:

```text
Dataset  ->  Rollout  ->  Rubric
  tasks      interaction   score
```

The same components can support several experiments:

- **Evaluation:** run tasks and aggregate rubric scores.
- **RL training:** use rubric scores as rewards.
- **Synthetic data:** keep useful high-scoring rollouts for later training.
- **Harness comparisons:** change the model or interaction setup and compare results.

I will build a small environment from scratch, then recreate it with a library to see what the abstraction provides.

---
## 2. Define a small coding dataset

Each task in this experiment contains a problem statement, a function name, and input-output test cases. I am keeping five toy problems small enough to inspect end to end. A larger evaluation could use a maintained coding benchmark or realistic repository tasks.

In [ ]:
PROBLEMS = [
    {
        "id": "add",
        "prompt": "Write a Python function `add(a, b)` that returns a + b.",
        "function_name": "add",
        "tests": [
            ("add(1, 2)", 3),
            ("add(-5, 5)", 0),
            ("add(0, 0)", 0),
            ("add(100, 200)", 300),
        ],
    },
    {
        "id": "is_even",
        "prompt": "Write a Python function `is_even(n)` that returns True if n is even, False otherwise.",
        "function_name": "is_even",
        "tests": [
            ("is_even(2)", True),
            ("is_even(3)", False),
            ("is_even(0)", True),
            ("is_even(-4)", True),
        ],
    },
    {
        "id": "reverse_string",
        "prompt": "Write a Python function `reverse_string(s)` that returns the reverse of string s.",
        "function_name": "reverse_string",
        "tests": [
            ("reverse_string('hello')", "olleh"),
            ("reverse_string('')", ""),
            ("reverse_string('a')", "a"),
            ("reverse_string('ab')", "ba"),
        ],
    },
    {
        "id": "fizzbuzz",
        "prompt": (
            "Write a Python function `fizzbuzz(n)` that returns 'Fizz' if n is divisible by 3, "
            "'Buzz' if divisible by 5, 'FizzBuzz' if divisible by both, else str(n)."
        ),
        "function_name": "fizzbuzz",
        "tests": [
            ("fizzbuzz(3)", "Fizz"),
            ("fizzbuzz(5)", "Buzz"),
            ("fizzbuzz(15)", "FizzBuzz"),
            ("fizzbuzz(7)", "7"),
        ],
    },
    {
        "id": "count_vowels",
        "prompt": "Write a Python function `count_vowels(s)` that returns the count of vowels (aeiouAEIOU) in s.",
        "function_name": "count_vowels",
        "tests": [
            ("count_vowels('hello')", 2),
            ("count_vowels('xyz')", 0),
            ("count_vowels('AEIOU')", 5),
            ("count_vowels('')", 0),
        ],
    },
]

print(f"Loaded {len(PROBLEMS)} problems.")
for p in PROBLEMS:
    print(f"  - {p['id']}: {len(p['tests'])} tests")


Loaded 5 problems.
  - add: 4 tests
  - is_even: 4 tests
  - reverse_string: 4 tests
  - fizzbuzz: 4 tests
  - count_vowels: 4 tests


---
## 3. Run generated code in a sandbox

I need to execute model-written code to check whether it passes the tests. This notebook uses a subprocess with a timeout as a simple demonstration. It is not a security boundary for untrusted code; production use needs an isolated sandbox such as a container or dedicated sandbox service.

For this prototype, I will check three basic failure cases:

1. Syntax errors
2. Runs that exceed the timeout
3. Valid code that returns the wrong result

In [ ]:
def run_code(code: str, test_call: str, expected: Any, timeout: int = 5) -> dict:
    """Run one generated-code test and capture its result in a subprocess.

    Returns dict with: passed (bool), error (str or None), output (str).
    """
    # Assemble a script that executes the candidate code and one test.
    # Use repr() so expected strings, booleans, and numbers stay valid Python.
    script = textwrap.dedent(f"""
        import sys
        try:
{textwrap.indent(code, ' ' * 12)}
        except Exception as e:
            print(f"SETUP_ERROR: {{type(e).__name__}}: {{e}}")
            sys.exit(1)

        try:
            result = {test_call}
            expected = {expected!r}
            if result == expected:
                print("PASS")
            else:
                print(f"FAIL: got {{result!r}}, expected {{expected!r}}")
        except Exception as e:
            print(f"RUNTIME_ERROR: {{type(e).__name__}}: {{e}}")
    """)

    with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
        f.write(script)
        path = f.name

    try:
        proc = subprocess.run(
            ['python', path],
            capture_output=True,
            text=True,
            timeout=timeout,
        )
        output = proc.stdout.strip() + proc.stderr.strip()
        passed = output.startswith("PASS")
        error = None if passed else output
        return {"passed": passed, "error": error, "output": output}
    except subprocess.TimeoutExpired:
        return {"passed": False, "error": "TIMEOUT", "output": ""}
    finally:
        os.unlink(path)


I will exercise the runner with correct code, an incorrect implementation, a syntax error, and an infinite loop. This checks whether the prototype reports each outcome as expected.

In [ ]:
# Check 1: expected output from a correct implementation.
print("Case 1 — correct code:")
print(run_code("def add(a, b):\n    return a + b", "add(2, 3)", 5))

# Check 2: detect an implementation that returns the wrong value.
print("\nCase 2 — wrong answer:")
print(run_code("def add(a, b):\n    return a - b", "add(2, 3)", 5))

# Check 3: report code that cannot be parsed.
print("\nCase 3 — syntax error:")
print(run_code("def add(a, b\n    return a + b", "add(2, 3)", 5))

# Check 4: stop code that runs beyond the time limit.
print("\nCase 4 — infinite loop:")
print(run_code("def add(a, b):\n    while True: pass\n    return a + b", "add(2, 3)", 5, timeout=2))


Case 1 — correct code:
{'passed': True, 'error': None, 'output': 'PASS'}

Case 2 — wrong answer:
{'passed': False, 'error': 'FAIL: got -1, expected 5', 'output': 'FAIL: got -1, expected 5'}

Case 3 — syntax error:
{'passed': False, 'error': 'File "/tmp/tmp54_thu_l.py", line 4\n    def add(a, b\n           ^\nSyntaxError: \'(\' was never closed', 'output': 'File "/tmp/tmp54_thu_l.py", line 4\n    def add(a, b\n           ^\nSyntaxError: \'(\' was never closed'}

Case 4 — infinite loop:
{'passed': False, 'error': 'TIMEOUT', 'output': ''}


---
## 4. Assemble the environment from scratch

I now have the three components and will connect them in one loop:

```text
Problem data -> model rollout -> generated code -> tests and rubric -> score
```

The `evaluate` function below runs each task, records a transcript, and returns the mean pass rate.

In [ ]:
def rollout(problem: dict, model: str = "google/gemini-2.0-flash-001") -> str:
    """Request a candidate solution from the model and return its response."""
    system = (
        "You are a Python coding assistant. "
        "When asked to write a function, respond with ONLY the function definition. "
        "No explanations, no markdown fences, just the code."
    )
    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": problem["prompt"]},
        ],
        temperature=0.0,
        max_tokens=300,
    )
    return resp.choices[0].message.content


def rubric(generated_code: str, problem: dict) -> dict:
    """Run the task tests and return their results with the pass rate."""
    results = []
    for test_call, expected in problem["tests"]:
        result = run_code(generated_code, test_call, expected)
        results.append((test_call, result["passed"], result.get("error")))
    pass_rate = sum(1 for _, passed, _ in results if passed) / len(results)
    return {"pass_rate": pass_rate, "results": results}


def evaluate(problems: list[dict], model: str) -> dict:
    """Run the rollout and scoring loop over my task set."""
    scores = []
    transcripts = []
    for p in problems:
        code = rollout(p, model=model)
        score = rubric(code, p)
        scores.append(score["pass_rate"])
        transcripts.append({
            "problem": p["id"],
            "code": code,
            "pass_rate": score["pass_rate"],
            "details": score["results"],
        })
        print(f"  {p['id']}: {score['pass_rate']:.0%} ({sum(1 for _, ok, _ in score['results'] if ok)}/{len(score['results'])} tests)")
    return {
        "mean_pass_rate": sum(scores) / len(scores),
        "transcripts": transcripts,
    }


In [ ]:
# Run my hand-built environment on the current task set.
print("Evaluating gemini-2.0-flash on our 5 problems:")
print("-" * 50)
result = evaluate(PROBLEMS, model="google/gemini-2.0-flash-001")
print("-" * 50)
print(f"\nOverall pass rate: {result['mean_pass_rate']:.1%}")


Evaluating gemini-2.0-flash on our 5 problems:
--------------------------------------------------
  add: 0% (0/4 tests)
  is_even: 0% (0/4 tests)
  reverse_string: 0% (0/4 tests)
  fizzbuzz: 0% (0/4 tests)
  count_vowels: 0% (0/4 tests)
--------------------------------------------------

Overall pass rate: 0.0%


### First environment run

The rollout, test runner, and rubric are now connected. I will inspect a transcript to see the generated code, per-test results, and score for one task.

In [ ]:
# Inspect one full rollout transcript and its test results.
import pprint
pprint.pp(result["transcripts"][0])


{'problem': 'add',
 'code': '```python\ndef add(a, b):\n  return a + b\n```',
 'pass_rate': 0.0,
 'details': [('add(1, 2)',
              False,
              'File "/tmp/tmphp33e4gu.py", line 4\n'
              '    ```python\n'
              '    ^\n'
              'SyntaxError: invalid syntax'),
             ('add(-5, 5)',
              False,
              'File "/tmp/tmp6so_82oc.py", line 4\n'
              '    ```python\n'
              '    ^\n'
              'SyntaxError: invalid syntax'),
             ('add(0, 0)',
              False,
              'File "/tmp/tmp7pri6_gv.py", line 4\n'
              '    ```python\n'
              '    ^\n'
              'SyntaxError: invalid syntax'),
             ('add(100, 200)',
              False,
              'File "/tmp/tmp8m4e4pbn.py", line 4\n'
              '    ```python\n'
              '    ^\n'
              'SyntaxError: invalid syntax')]}


---
## 5. Probe the rubric for brittle inputs

The initial rollout asks for code only, but a model may still return Markdown fences or explanatory prose. I will remove the strict system prompt and see whether the current rubric handles that response format.

In [ ]:
# Remove the code-only system prompt to probe how the parser handles prose.
def chatty_rollout(problem: dict, model: str = "google/gemini-2.0-flash-001") -> str:
    resp = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": problem["prompt"]}],
        temperature=0.0,
        max_tokens=300,
    )
    return resp.choices[0].message.content

# Try the altered prompt on one task.
chatty_code = chatty_rollout(PROBLEMS[0])
print("Model output:\n")
print(chatty_code)
print("\n" + "=" * 50)
print("Score with naive rubric:")
print(rubric(chatty_code, PROBLEMS[0]))


Model output:

```python
def add(a, b):
  """
  This function takes two numbers as input and returns their sum.

  Args:
    a: The first number.
    b: The second number.

  Returns:
    The sum of a and b.
  """
  return a + b
```

Score with naive rubric:
{'pass_rate': 0.0, 'results': [('add(1, 2)', False, 'File "/tmp/tmpl1_oln7h.py", line 4\n    ```python\n    ^\nSyntaxError: invalid syntax'), ('add(-5, 5)', False, 'File "/tmp/tmpwqd0ueg9.py", line 4\n    ```python\n    ^\nSyntaxError: invalid syntax'), ('add(0, 0)', False, 'File "/tmp/tmpot7l5h1j.py", line 4\n    ```python\n    ^\nSyntaxError: invalid syntax'), ('add(100, 200)', False, 'File "/tmp/tmpx6bcu4jg.py", line 4\n    ```python\n    ^\nSyntaxError: invalid syntax')]}


The model can provide a correct function inside a Markdown code fence, while the naive runner treats the entire response as Python and reports a syntax error. This is a parser problem: before scoring a response, the environment needs to extract the structured content it expects.

In [ ]:
import re

def extract_code(text: str) -> str:
    """Extract candidate Python code from a response with fences or prose."""
    # First look for a fenced Python code block.
    fence_match = re.search(r"```(?:python)?\s*\n(.*?)```", text, re.DOTALL)
    if fence_match:
        return fence_match.group(1).strip()
    # Otherwise, skip leading prose and look for a line that starts code.
    lines = text.strip().split("\n")
    # Start at the first recognizable definition, import, or assignment.
    for i, line in enumerate(lines):
        if re.match(r"^(def |class |import |from |\w+\s*=)", line.strip()):
            return "\n".join(lines[i:])
    return text.strip()


def robust_rubric(model_response: str, problem: dict) -> dict:
    """Extract code before running the tests in my rubric."""
    code = extract_code(model_response)
    results = []
    for test_call, expected in problem["tests"]:
        result = run_code(code, test_call, expected)
        results.append((test_call, result["passed"], result.get("error")))
    pass_rate = sum(1 for _, passed, _ in results if passed) / len(results)
    return {"pass_rate": pass_rate, "results": results, "extracted": code}


# Re-score the chatty response after extracting its code.
print("Score with robust rubric:")
print(robust_rubric(chatty_code, PROBLEMS[0]))


Score with robust rubric:
{'pass_rate': 1.0, 'results': [('add(1, 2)', True, None), ('add(-5, 5)', True, None), ('add(0, 0)', True, None), ('add(100, 200)', True, None)], 'extracted': 'def add(a, b):\n  """\n  This function takes two numbers as input and returns their sum.\n\n  Args:\n    a: The first number.\n    b: The second number.\n\n  Returns:\n    The sum of a and b.\n  """\n  return a + b'}


### Observation: parsing is part of the reward pipeline

A reliable environment needs to parse model output before scoring it. I separate those two steps in this notebook so I can see how extraction errors affect the score.

---
## 6. Rebuild the environment with Verifiers

I will recreate the same task loop with the [Verifiers library](https://github.com/PrimeIntellect-ai/verifiers). This lets me compare my hand-built version with library support for parsers, composable rubrics, environment classes, and evaluation.

In [ ]:
!pip install -q verifiers


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.8/645.8 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 753.6/753.6 kB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.1/209.1 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 826.2/826.2 kB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 841.0/841.0 kB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.2/244.2 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 727.0/727.0 kB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.7/310.7 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.4/63.4 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.3/324.3 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 95.3 MB/s eta 0:

In [ ]:
import verifiers as vf
from datasets import Dataset
print(f"Verifiers version: {vf.__version__ if hasattr(vf, '__version__') else 'installed'}")


Verifiers version: 0.1.14


### 6.1 Convert tasks to a Hugging Face dataset

I will place each problem statement and function name in dataset columns, with the test cases stored as task metadata for the reward function.

In [ ]:
import json

dataset_rows = []
for p in PROBLEMS:
    dataset_rows.append({
        "question": p["prompt"],
        "answer": p["function_name"],
        "info": {
            "function_name": p["function_name"],
            "tests_json": json.dumps(p["tests"]),
            "id": p["id"],
        },
    })

vf_dataset = Dataset.from_list(dataset_rows)
print(vf_dataset)
print("\nFirst row:")
print(vf_dataset[0])

Dataset({
    features: ['question', 'answer', 'info'],
    num_rows: 5
})

First row:
{'question': 'Write a Python function `add(a, b)` that returns a + b.', 'answer': 'add', 'info': {'function_name': 'add', 'id': 'add', 'tests_json': '[["add(1, 2)", 3], ["add(-5, 5)", 0], ["add(0, 0)", 0], ["add(100, 200)", 300]]'}}


### 6.2 Define the Verifiers rubric

The rubric combines two signals: the fraction of tests passed and whether the extracted code parses. I give test correctness more weight so parseability alone cannot dominate the score.

In [ ]:
def code_passes_tests(prompt, completion, answer, state, info=None, **kwargs) -> float:
    """Return the fraction of task tests passed, from zero to one."""
    response = completion[-1]["content"] if isinstance(completion, list) else str(completion)
    code = extract_code(response)
    if info is None:
        return 0.0
    tests = info["tests"]
    passed = 0
    for test_call, expected in tests:
        r = run_code(code, test_call, expected)
        if r["passed"]:
            passed += 1
    return passed / len(tests)


def code_is_parseable(prompt, completion, answer, state, info=None, **kwargs) -> float:
    """Return one when the extracted code parses, otherwise zero."""
    response = completion[-1]["content"] if isinstance(completion, list) else str(completion)
    code = extract_code(response)
    try:
        compile(code, "<rubric>", "exec")
        return 1.0
    except SyntaxError:
        return 0.0


# Weight task correctness more heavily than code parseability.
rubric = vf.Rubric(
    funcs=[code_passes_tests, code_is_parseable],
    weights=[5.0, 1.0],
)
print("Rubric created with 2 reward functions.")


Rubric created with 2 reward functions.


### 6.3 Build the single-turn environment

I will pass the task dataset, system prompt, and rubric to `SingleTurnEnv` and check that the environment is constructed.

In [ ]:
SYSTEM_PROMPT = (
    "You are a Python coding assistant. Write a function that solves the user's request. "
    "Respond with the function definition only — no markdown fences, no explanation."
)

env = vf.SingleTurnEnv(
    dataset=vf_dataset,
    system_prompt=SYSTEM_PROMPT,
    rubric=rubric,
)
print("Environment built:", type(env).__name__)


Map:   0%|          | 0/5 [00:00<?, ? examples/s]

Environment built: SingleTurnEnv


### 6.4 Evaluate the single-turn environment

I will run the same tasks through Verifiers using an OpenAI-compatible client. This gives me a way to compare the library evaluation loop with the hand-built version.

In [ ]:
import os
from google.colab import userdata

os.environ["OPENROUTER_API_KEY"] = userdata.get('OPENROUTER_API_KEY')

# Confirm that the API key is available before evaluation.
assert os.environ["OPENROUTER_API_KEY"].startswith("sk-or-"), "key not set"

from verifiers.clients import ClientConfig

vf_config = ClientConfig(
    client_type="openai_chat_completions",
    api_key_var="OPENROUTER_API_KEY",
    api_base_url="https://openrouter.ai/api/v1",
)

results = await env.evaluate(
    client=vf_config,
    model="google/gemini-2.0-flash-001",
    num_examples=5,
    rollouts_per_example=1,
    sampling_args={"max_tokens": 400},
)

print(results)

eval_dataset is not set, falling back to train dataset
Processing 5 groups (5 total rollouts): 100%|██████████| 5/5 [00:01<00:00,  3.45it/s, reward=1]

{'outputs': [{'example_id': 0, 'prompt': [SystemMessage(role='system', content="You are a Python coding assistant. Write a function that solves the user's request. Respond with the function definition only — no markdown fences, no explanation."), UserMessage(role='user', content='Write a Python function `add(a, b)` that returns a + b.')], 'completion': [AssistantMessage(role='assistant', content='```python\ndef add(a, b):\n  """\n  This function adds two numbers a and b and returns the sum.\n  """\n  return a + b\n```\n', reasoning_content=None, thinking_blocks=None, tool_calls=None)], 'answer': 'add', 'info': {'function_name': 'add', 'id': 'add', 'tests_json': '[["add(1, 2)", 3], ["add(-5, 5)", 0], ["add(0, 0)", 0], ["add(100, 200)", 300]]'}, 'reward': 1.0, 'error': None, 'timing': {'start_time': 1778182946.090946, 'setup': {'start': 1778182946.090989, 'end': 1778182946.090991, 'duration': 1.9073486328125e-06}, 'generation': {'start': 1778182946.0909836, 'end': 1778182946.896391, 'dur

### What the library version adds

The same task set and scoring idea now use a reusable environment object. The library also provides support for evaluation, training integration, and sharing environments. I can build on one definition across several experiments.

---
## 7. Add multi-turn feedback

In a multi-turn rollout, the model sees test feedback and can revise its code. I will first implement this loop directly so I can inspect each step, then sketch how a `MultiTurnEnv` can represent the same interaction.

```text
Prompt -> generated code -> tests
                      | all pass: end
                      | failures: send feedback
Feedback and history -> revised code -> tests -> ...
```

In [ ]:
def multi_turn_rollout(
    problem: dict,
    model: str = "google/gemini-2.0-flash-001",
    max_turns: int = 3,
) -> dict:
    """Request code, test it, and return failures for another attempt."""
    messages = [
        {"role": "system", "content": (
            "You are a Python coding assistant. Write a function that solves the user's request. "
            "If the user reports test failures, analyze them and produce a corrected function."
        )},
        {"role": "user", "content": problem["prompt"]},
    ]

    turn_history = []

    for turn in range(1, max_turns + 1):
        resp = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=0.0,
            max_tokens=400,
        )
        model_output = resp.choices[0].message.content
        messages.append({"role": "assistant", "content": model_output})

        code = extract_code(model_output)
        results = []
        for test_call, expected in problem["tests"]:
            r = run_code(code, test_call, expected)
            results.append((test_call, r["passed"], r.get("error")))

        n_pass = sum(1 for _, ok, _ in results if ok)
        pass_rate = n_pass / len(results)
        turn_history.append({"turn": turn, "pass_rate": pass_rate, "code": code})

        if pass_rate == 1.0:
            return {"final_pass_rate": 1.0, "turns_used": turn, "history": turn_history}

        # Turn failed tests into feedback for the next model turn.
        failures = [(call, err) for call, ok, err in results if not ok]
        feedback_lines = [f"Your code passed {n_pass}/{len(results)} tests. Failures:"]
        for call, err in failures[:3]:   # Limit feedback to three failures per turn.
            feedback_lines.append(f"  - {call} → {err}")
        feedback_lines.append("Please fix and provide the corrected function.")
        messages.append({"role": "user", "content": "\n".join(feedback_lines)})

    return {"final_pass_rate": pass_rate, "turns_used": max_turns, "history": turn_history}


I will run the iterative loop on a task with edge cases that may expose an incomplete first attempt, then inspect the pass rate and code from each turn.

In [ ]:
# Add edge cases to a task so I can observe a possible first-turn failure.
tricky_problem = {
    "id": "edge_count_vowels",
    "prompt": (
        "Write a Python function `count_vowels(s)` that returns the count of vowels in s. "
        "Vowels are aeiouAEIOU. The function must also handle the case where 'y' is a vowel "
        "if and only if it is NOT at the start or end of the string."
    ),
    "function_name": "count_vowels",
    "tests": [
        ("count_vowels('hello')", 2),
        ("count_vowels('rhythm')", 1),       # The middle y counts.
        ("count_vowels('yellow')", 2),       # The starting y does not count; e and o do.
        ("count_vowels('hairy')", 2),        # The ending y does not count; a and i do.
        ("count_vowels('lyly')", 1),         # Only the middle y counts.
    ],
}

print(f"Running multi-turn rollout (max 3 turns) on: {tricky_problem['id']}")
print("=" * 60)
trace = multi_turn_rollout(tricky_problem, max_turns=3)

for h in trace["history"]:
    print(f"\n--- Turn {h['turn']}: pass rate {h['pass_rate']:.0%} ---")
    print(h["code"][:400])

print("\n" + "=" * 60)
print(f"Final: {trace['final_pass_rate']:.0%} after {trace['turns_used']} turn(s)")


Running multi-turn rollout (max 3 turns) on: edge_count_vowels

--- Turn 1: pass rate 100% ---
def count_vowels(s):
    """Counts the number of vowels (aeiouAEIOU) in the string s.
    'y' is a vowel if and only if it is NOT at the start or end of the string.
    """
    vowels = "aeiouAEIOU"
    count = 0
    for i, char in enumerate(s):
        if char in vowels:
            count += 1
        elif char in "yY" and 0 < i < len(s) - 1:
            count += 1
    return count

Final: 100% after 1 turn(s)


In [ ]:
tricky_problem = {
    "id": "edge_count_vowels_v2",
    "prompt": (
        "Write `count_vowels(s)` that returns the count of vowels in s. "
        "Rules: vowels are aeiouAEIOU. 'y' is a vowel ONLY when NOT at the start or end. "
        "If s contains ANY digit, return -1. If s is None, return 0 (not an error). "
        "Apostrophes count as silent — skip them entirely (don't break adjacency for the y rule)."
    ),
    "function_name": "count_vowels",
    "tests": [
        ("count_vowels('hello')", 2),
        ("count_vowels('rhythm')", 1),
        ("count_vowels('yellow')", 2),
        ("count_vowels('hairy')", 2),
        ("count_vowels('hello123')", -1),
        ("count_vowels(None)", 0),
        ("count_vowels(\"y'all\")", 0),
    ],
}

### What I learned from the iterative run

The trajectory is code, feedback, revised code, and more feedback. The final pass rate is one possible reward; I could also include turn count if efficiency matters.

Design questions for my next iteration:

- **Reward shaping:** should a solution that passes on the first turn score higher than one that passes later?
- **Feedback quality:** should I return raw test failures or convert them into hints?
- **Context growth:** how should I manage longer histories in extended tasks?

A Verifiers implementation can subclass `vf.MultiTurnEnv` and use persistent state to track the task and turn. This sketch shows the shape of the interaction:

```python
class CodeFixerEnv(vf.MultiTurnEnv):
    async def env_response(self, messages, state, **kwargs):
        # I inspect the latest model response and run the task tests.
        latest = messages[-1]["content"]
        code = extract_code(latest)
        # I would build feedback from the failed tests here.
        if all_passed:
            return [], state
        return [{"role": "user", "content": feedback}], state

    async def is_completed(self, messages, state, **kwargs):
        return state.get("all_passed", False) or state.get("turn", 0) >= self.max_turns
```

The exact subclass API can vary by library version, so I would check the installed documentation before turning this sketch into a runnable implementation.

---
## 8. Compare models with the same environment

An environment can act as an evaluation harness: each model sees the same tasks and receives the same scoring. I will compare several available models on the toy problem set and rank their mean pass rates.

In [ ]:
def score_code(generated_code: str, problem: dict) -> dict:
    code = extract_code(generated_code)   # Parse the response before running the tests.
    results = []
    for test_call, expected in problem["tests"]:
        result = run_code(code, test_call, expected)
        results.append((test_call, result["passed"], result.get("error")))
    pass_rate = sum(1 for _, passed, _ in results if passed) / len(results)
    return {"pass_rate": pass_rate, "results": results}


def evaluate(problems, model):
    scores, transcripts = [], []
    for p in problems:
        code = rollout(p, model=model)
        s = score_code(code, p)
        scores.append(s["pass_rate"])
        transcripts.append({"problem": p["id"], "code": code, "pass_rate": s["pass_rate"], "details": s["results"]})
        print(f"  {p['id']}: {s['pass_rate']:.0%} ({sum(1 for _, ok, _ in s['results'] if ok)}/{len(s['results'])} tests)")
    return {"mean_pass_rate": sum(scores) / len(scores), "transcripts": transcripts}

In [ ]:
# Compare several available models on the same task set and rubric.
MODELS_TO_COMPARE = [
    "google/gemini-2.0-flash-001",
    "openai/gpt-4o-mini",
    "qwen/qwen-2.5-coder-32b-instruct",
]

comparison = {}
for model_id in MODELS_TO_COMPARE:
    print(f"\n{'=' * 50}")
    print(f"Evaluating: {model_id}")
    print('=' * 50)
    try:
        result = evaluate(PROBLEMS, model=model_id)
        comparison[model_id] = result["mean_pass_rate"]
    except Exception as e:
        print(f"  ERROR: {e}")
        comparison[model_id] = None

print("\n\n" + "=" * 50)
print("FINAL RANKING")
print("=" * 50)
ranked = sorted(
    [(m, s) for m, s in comparison.items() if s is not None],
    key=lambda x: -x[1]
)
for i, (model, score) in enumerate(ranked, 1):
    print(f"  {i}. {model}: {score:.1%}")



Evaluating: google/gemini-2.0-flash-001
  add: 100% (4/4 tests)
  is_even: 100% (4/4 tests)
  reverse_string: 100% (4/4 tests)
  fizzbuzz: 100% (4/4 tests)
  count_vowels: 100% (4/4 tests)

Evaluating: openai/gpt-4o-mini
  add: 100% (4/4 tests)
  is_even: 100% (4/4 tests)
  reverse_string: 100% (4/4 tests)
  fizzbuzz: 0% (0/4 tests)
  count_vowels: 0% (0/4 tests)

Evaluating: qwen/qwen-2.5-coder-32b-instruct
  add: 100% (4/4 tests)
  is_even: 0% (0/4 tests)
  reverse_string: 0% (0/4 tests)
  fizzbuzz: 0% (0/4 tests)
  count_vowels: 0% (0/4 tests)


FINAL RANKING
  1. google/gemini-2.0-flash-001: 100.0%
  2. openai/gpt-4o-mini: 60.0%
  3. qwen/qwen-2.5-coder-32b-instruct: 20.0%


### Why I am using one environment for comparisons

The tasks, rollout, and rubric are held constant, so the resulting scores are directly comparable within this small experiment. From here I could scale the task set, use repository-level checks, or save high-scoring rollouts as candidate SFT examples.

The same environment definition can support evaluation, training rewards, and data collection.

---
## Project notes and next steps

### What I built

- A coding task environment from scratch with a dataset, rollout, and rubric.
- A sandbox prototype that reports test results and timeouts.
- A parser that extracts code before scoring it.
- A Verifiers version with composable rewards and an evaluation interface.
- A multi-turn loop that returns test feedback to the model.
- A small cross-model comparison using the same tasks and scoring.

### What I want to improve next

1. Replace toy tasks with a larger, held-out coding set.
2. Run generated code in a properly isolated sandbox before using untrusted outputs.
3. Test parser behavior across fenced code, prose, and malformed responses.
4. Compare first-turn and multi-turn rewards, including the cost of extra turns.
5. Measure model differences over more tasks and repeat runs.

### References I am using

- **Verifiers:** [GitHub](https://github.com/PrimeIntellect-ai/verifiers) and [documentation](https://docs.primeintellect.ai/verifiers)
- **Environments Hub:** [Prime Intellect](https://app.primeintellect.ai/dashboard/environments)
- **OpenEnv:** [GitHub](https://github.com/meta-pytorch/OpenEnv)
- **Environments Hub walkthrough:** [Hugging Face blog](https://huggingface.co/blog/anakin87/environments-hub)
- **INTELLECT-3 technical report:** [arXiv](https://arxiv.org/abs/2512.16144)
- **Verifiers paper and citation:** [GitHub](https://github.com/PrimeIntellect-ai/verifiers#citation)